# Graded Activity: Let's Create a Bag of Words model for Sarcasm Detection
In this activity, you will create a Bag of Words model to detect sarcasm in text data. This is __not__ a good choice for a sarcasm detection model, but it is a good exercise to understand how Bag of Words works.

Fill me in here.

So let's get started!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

The [include command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

In [1]:
include("Include.jl");

In addition to standard Julia libraries, we'll also use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl), check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types and data used in this material.

### Data
Let's load a public dataset of headlines that have been curated as either __sarcastic__ or __not sarcastic__. The dataset we'll use is [publically available on Kaggle](https://www.kaggle.com/datasets/rmisra/news-headlines-dataset-for-sarcasm-detection) and is also discussed in the publications:
1. Misra, Rishabh and Prahal Arora. "Sarcasm Detection using News Headlines Dataset." AI Open (2023).
2. Misra, Rishabh and Jigyasa Grover. "Sculpting Data for ML: The first act of Machine Learning." ISBN 9798585463570 (2021).

We've packaged the sarcasm dataset in [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl). We'll load the dataset using [the `MySarcasmCorpus(...)` method](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/data/#VLDataScienceMachineLearningPackage.MySarcasmCorpus) which returns [a `MySarcasmRecordCorpusModel` instance](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/types/#VLDataScienceMachineLearningPackage.MySarcasmRecordCorpusModel) with the fields:
* The `records::Dict{Int, MySarcasmRecordModel}` field holds the original records data as a dictionary, where the keys of the dictionary correspond to the headline index, and the values are [instances of the `MySarcasmRecordModel` type](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/types/#VLDataScienceMachineLearningPackage.MySarcasmRecordModel). Each record has the following fields:
    * `issarcastic`: has a value of `1` if the record is sarcastic; otherwise, `0.`
    * `headline`: the headline of the article, unstructured text
    * `article_link`: link to the original news article. Useful in collecting supplementary data

* The `tokens::Dict{String, Int64}` field holds the vocabulary computed over the __entire dataset__ as a dictionary, where the dictionary's keys are the tokens (words) and the values of the index of the word. We assemble the `tokens` dictionary in alphabetical order. 
* The `inverse::Dict{Int64, String}` field is the inverse of the `tokens` dictionary, where the keys are the token indexes and the values are the tokens (words).

Let's call the `MySarcasmCorpus(...)` method to load the sarcasm dataset and assign it to the `corpusmodel:MySarcasmRecordCorpusModel` variable. 

In [2]:
corpusmodel = MySarcasmCorpus(); # this loads the corpus model, which contains the vocabulary and tokenization information


### Compute Maximum Pad Length
Before we start, we need to compute the maximum pad length for the Bag of Words model. This will ensure that all input sequences are of the same length, which is generally a nice to have, and a requirement in some cases, for many machine learning models. 

Iterate through each headline using [a for-loop](https://docs.julialang.org/en/v1/manual/control-flow/#For-Loops-1), compute its size using [the `length(...)` method](https://docs.julialang.org/en/v1/base/collections/#Base.length), and then save this length.  If the test length is longer than we've seen before, this becomes the new maximum pad length.


In [3]:
max_pad_length = let

    # initialize -
    number_of_records = corpusmodel.records |> length; # how many records do we have?
    max_pad_length = 0; # initialize: we have 0 length
    
    # test the length of each headline
    for i ∈ 1:number_of_records
        test_record_length = tokenize(corpusmodel.records[i].headline, corpusmodel.tokens) |> length; # tokenize, and calc the number of tokens
        if (test_record_length > max_pad_length)
            max_pad_length = test_record_length; # we've found a new longest headline!
            println("Found a new longest headline: $(i) with length: $(max_pad_length)"); # show the record number and length
        end
    end
    max_pad_length
end;

Found a new longest headline: 1 with length: 10
Found a new longest headline: 2 with length: 15
Found a new longest headline: 11 with length: 16
Found a new longest headline: 14 with length: 18
Found a new longest headline: 37 with length: 20
Found a new longest headline: 97 with length: 21
Found a new longest headline: 106 with length: 22
Found a new longest headline: 189 with length: 23
Found a new longest headline: 584 with length: 24
Found a new longest headline: 1238 with length: 25
Found a new longest headline: 1450 with length: 26
Found a new longest headline: 2147 with length: 31
Found a new longest headline: 7303 with length: 153


### Balance
Next, let's count the number of sarcastic and unsarcastic samples in the dataset. This will help us understand the balance of the dataset and whether we need to take any steps to address class imbalance.

In [4]:
number_of_sarchastic_records = findall(r -> r.issarcastic == 1, corpusmodel.records) |> length;
number_of_unsarcastic_records = findall(r -> r.issarcastic == 0, corpusmodel.records) |> length;

## Task 1: Tokenize the Headlines
In this task, we'll tokenize the headlines in the sarcasm dataset. Tokenization is the process of splitting text into individual words or tokens. We'll use [the `tokenize(...)` method exported by the `VLDataScienceMachineLearningPackage.jl` package](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/text/#VLDataScienceMachineLearningPackage.tokenize) to perform this task. 

We'll store the original headlines, the tokenized headlines and the headline labels [as NamedTuple instances](https://docs.julialang.org/en/v1/base/base/#Core.NamedTuple) in the `tokenized_headlines_vector::Vector{NamedTuple}` variable. Each Named Tuple will have the following fields:
- `headline::String`: the original headline text
- `tokens::Vector{Int64}`: the tokenized headline, where each token is represented by its index in the vocabulary
- `issarcastic::Int64`: the label of the headline, where `1` indicates sarcasm and `0` indicates no sarcasm

We should have a vector of Named Tuples, where each Named Tuple corresponds to a headline in the sarcasm dataset.

In [5]:
tokenized_headlines_vector = let

    # initialize -
    number_of_records = corpusmodel.records |> length; # how many records do we have?
    tokenized_headlines_vector = Vector{NamedTuple}(); # initialize the vector of NamedTuples

    # tokenize each record, store result in the dictionary
    for i ∈ 1:number_of_records

        model = corpusmodel.records[i]; # get the record
        headline = model.headline; # get the headline
        label = model.issarcastic; # get the label

        tokens = tokenize(headline, corpusmodel.tokens,  pad = max_pad_length); # tokenize, and pad to max length

        data_tuple = (headline = headline, issarcastic = label, tokens = tokens); # create a NamedTuple with the headline, label, and tokens
        push!(tokenized_headlines_vector, data_tuple); # add the NamedTuple to the
    end

    tokenized_headlines_vector;
end;

### Checkpoint
Ok, so we have processed the headlines and created a vector of Named Tuples. Each Named Tuple contains the original headline, the tokenized headline, and the label indicating whether the headline is sarcastic or not. Let's check that we are going what we think we are doing by writing a few unit tests to check the tokenized records and lables correspond to the original records.

We'll use [the `Test.jl` package](https://docs.julialang.org/en/v1/stdlib/Test/) to write __unit tests__ for our tokenization pipeline. The [`Test.jl` package](https://docs.julialang.org/en/v1/stdlib/Test/) provides a framework for writing and running single and multiple tests in Julia, including support for assertions, test cases, and test suites.


In [6]:
let

    

    @testset "Tokenization Pipeline Tests" begin
        @test length(tokenized_headlines_vector) == length(corpusmodel.records) # check that we have the same number of records
        @test all(map(x -> x.issarcastic in (0, 1), tokenized_headlines_vector)) # check that all labels are either 0 or 1
        @test all(map(x -> length(x.tokens) == (max_pad_length + 1), tokenized_headlines_vector)) # check that all tokenized headlines are padded to the same length

        # pick a random headline to check -
        random_index = rand(1:length(tokenized_headlines_vector));
        random_headline = tokenized_headlines_vector[random_index].headline;
        random_tokens = tokenized_headlines_vector[random_index].tokens;
        inverse = corpusmodel.inverse; # inverse mapping of tokens to words
        
        # Convert the tokens back to a headline, stripingout the control tokens
        tmp = Vector{String}();
        for i ∈ eachindex(random_tokens)
            token = random_tokens[i];
            if inverse[token] == "<pad>" || inverse[token] == "<eos>" || inverse[token] == "<unk>" || inverse[token] == "<bos>"
                continue; # skip padding and unknown tokens
            else
                push!(tmp, inverse[token]); # add the token to the vector
            end
        end
        reconstructed_headline = join(tmp, " "); # join the tokens into a string
        @test reconstructed_headline == random_headline # check that the reconstructed headline matches the original headline
    end
end;

Test Summary:               | Pass  Total  Time
Tokenization Pipeline Tests |    4      4  0.4s


## Task 2: Create Sarcastic and Unsarcastic Feature Vectors
In this task, we will create two Bag of Words models, one for sarcastic samples and one for unsarcastic samples. We will split the sarcastic samples into a testing set and a control set. The control set will be used to create a Bag of Words model that we can compare against the sarcastic Bag of Words model, i.e., to compute a self-similarity score between sarcastic samples.

> __Strategy__: Let's split the `tokenized_headlines_vector` into four vectors, two for sarcastic samples (control and test) and the other two for unsarcastic (control and test) samples. We'll use [the `filter(...)` method](https://docs.julialang.org/en/v1/base/collections/#Base.filter) to filter the vector based on the `issarcastic` field. Oncem we have these four vectors, we'll combine the control and test vectors into a single vector for each Bag of Words model (using Term frequency–inverse document frequency, or TF-IDF).

First, specify what fraction of each population will be used for the control set. We'll use a 50% split for both sarcastic and unsarcastic samples, but you can change this to any value between 0 and 1.

In [7]:
fraction_sarcastic_control = 0.5; # fraction of sarcastic samples to use for control set
fraction_unsarcastic_control = 0.5; # fraction of unsarcastic samples to use for control set

Next, we will filter the `tokenized_headlines_vector` to create the test and control sets for the sarcastic samples. Update me with a description of the code you wrote to do this.

In [8]:
sarchastic_samples_test, sarchastic_samples_control = let

    # initialize -
    sarchastic_sample_test = Vector{NamedTuple}(); # vector for test samples
    sarchastic_sample_control = Vector{NamedTuple}(); # vector for control samples
    number_of_control_records = (number_of_sarchastic_records)*fraction_sarcastic_control |> floor |> Int; # how many control records do we have?

    # what is the index vector for the sarcastic samples?
    index_vector_sarcastic_samples = findall(r -> r.issarcastic == 1, tokenized_headlines_vector) |> collect; # get the indices of the sarcastic samples

    # create a shuffled index vector -
    shuffled_index_vector = shuffle(index_vector_sarcastic_samples);
    for i ∈ eachindex(shuffled_index_vector)

        # get the record index -
        record_index = shuffled_index_vector[i];

        # get the record -
        record = tokenized_headlines_vector[record_index];

        # add to the control or test set -
        if i <= number_of_control_records
            push!(sarchastic_sample_control, record); # add to control set
        else
            push!(sarchastic_sample_test, record); # add to test set
        end
    end

    sarchastic_sample_test, sarchastic_sample_control;
end;

We can do the same for the unsarcastic samples.

In [9]:
unsarcastic_samples_test, unsarcastic_samples_control = let

    # initialize -
    unsarcastic_sample_test = Vector{NamedTuple}(); # vector for test samples
    unsarcastic_sample_control = Vector{NamedTuple}(); # vector for control samples
    number_of_control_records = (number_of_unsarcastic_records)*fraction_unsarcastic_control |> floor |> Int; # how many control records do we have?

    # what is the index vector for the unsarcastic samples?
    index_vector_unsarcastic_samples = findall(r -> r.issarcastic == 0, tokenized_headlines_vector) |> collect; # get the indices of the unsarcastic samples

    # create a shuffled index vector -
    shuffled_index_vector = shuffle(index_vector_unsarcastic_samples);
    for i ∈ eachindex(shuffled_index_vector)

        # get the record index -
        record_index = shuffled_index_vector[i];

        # get the record -
        record = tokenized_headlines_vector[record_index];

        # add to the control or test set -
        if i <= number_of_control_records
            push!(unsarcastic_sample_control, record); # add to control set
        else
            push!(unsarcastic_sample_test, record); # add to test set
        end
    end

    (unsarcastic_sample_test, unsarcastic_sample_control);
end;

In [26]:
unsarcastic_samples_test

7493-element Vector{NamedTuple}:
 (headline = "why im going to let my daughter fail math", issarcastic = false, tokens = [29662, 29029, 13330, 11517, 26824, 15401, 17675, 7117, 9817, 16458  …  29665, 29665, 29665, 29665, 29665, 29665, 29665, 29665, 29665, 29663])
 (headline = "carrie fisher cant stop swearing during star wars the force awakens live telecast", issarcastic = false, tokens = [29662, 4692, 10315, 4556, 25421, 25980, 8624, 25199, 28695, 26532  …  29665, 29665, 29665, 29665, 29665, 29665, 29665, 29665, 29665, 29663])
 (headline = "the 7th heaven cast reunites for the first time in 8 years", issarcastic = false, tokens = [29662, 26532, 816, 12463, 4752, 22328, 10601, 26532, 10295, 26761  …  29665, 29665, 29665, 29665, 29665, 29665, 29665, 29665, 29665, 29663])
 (headline = "australian police charge vatican treasurer over historical sexual assaults", issarcastic = false, tokens = [29662, 2436, 20178, 5023, 28209, 27174, 18981, 12728, 23758, 2256  …  29665, 29665, 29665, 29665,

### Consensus Vectors
Now that we have the sarcastic and unsarcastic samples split into test and control sets, let's build consensus vectors for each set. 

> __Consensus vectors__ are vectors that contain are computed by calculating the averagev value of a feature across all samples in a set. In this case, we will create a consensus vector for each set of sarcastic and unsarcastic samples. We'll first compute a fatuer meatric where the samples are on the rows, and t6yhe feature sare on tghe columns. rhen we'll sum each feature across all samples in the set and divide by the number of samples to get the average value for each feature.

We'll use the [the `featurehashing(...)` method](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/text/#VLDataScienceMachineLearningPackage.featurehashing) to create a consenssus feature vector for each set. We'll save the resulting arrays as $\mathbf{F}_{1}, \mathbf{F}_{2}, \mathbf{F}_{3}, \mathbf{F}_{4}$, where $\mathbf{F}_{1}$ is the sarcastic test set, $\mathbf{F}_{2}$ is the sarcastic control set, $\mathbf{F}_{3}$ is the unsarcastic test set, and $\mathbf{F}_{4}$ is the unsarcastic control set.

In [38]:
F₁,F₂,F₃,F₄ = let

    # initialize -
    d = 20; # d is the number of tokens in the vocabulary, plus one for the padding token
    F₁ = zeros(length(sarchastic_samples_test), d); # initialize the sarcastic test set feature matrix
    F₂ = zeros(length(sarchastic_samples_control), d); # initialize the sarcastic control set feature matrix
    F₃ = zeros(length(unsarcastic_samples_test), d); # initialize the unsarcastic test set feature matrix
    F₄ = zeros(length(unsarcastic_samples_control), d); # initialize the unsarcastic control set feature matrix

    # F₁: sarcastic test set -
    for i ∈ eachindex(sarchastic_samples_test)
        tokens = sarchastic_samples_test[i].tokens; # get the tokens
        F₁[i, :] = featurehashing(tokens, d = d, algorithm = SignedFeatureHashing()); # signed feature hashing
    end

    # F₂: sarcastic control set -
    for i ∈ eachindex(sarchastic_samples_control)
        tokens = sarchastic_samples_control[i].tokens; # get the tokens
        F₂[i, :] = featurehashing(tokens, d = d, algorithm = SignedFeatureHashing()); # signed feature hashing
    end

    # F₃: unsarcastic test set -
    for i ∈ eachindex(unsarcastic_samples_test)
        tokens = unsarcastic_samples_test[i].tokens; # get the tokens
        F₃[i, :] = featurehashing(tokens, d = d, algorithm = SignedFeatureHashing()); # signed feature hashing
    end

    # F₄: unsarcastic control set -
    for i ∈ eachindex(unsarcastic_samples_control)
        tokens = unsarcastic_samples_control[i].tokens; # get the tokens
        F₄[i, :] = featurehashing(tokens, d = d, algorithm = SignedFeatureHashing()); # signed feature hashing
    end

    (F₁,F₂,F₃,F₄);
end;

In [40]:
F₁

6817×20 Matrix{Float64}:
 145.0  -1.0  0.0   0.0  1.0  -1.0  0.0  …  0.0   0.0  1.0   0.0  1.0  -1.0
 143.0  -1.0  0.0   0.0  3.0  -2.0  0.0     1.0   0.0  1.0   0.0  0.0   0.0
 140.0   0.0  0.0  -1.0  1.0   0.0  0.0     1.0  -1.0  1.0  -1.0  1.0   0.0
 141.0   0.0  2.0  -1.0  1.0   0.0  0.0     1.0  -1.0  0.0  -1.0  2.0  -1.0
 140.0  -2.0  0.0  -1.0  1.0   0.0  1.0     0.0   0.0  0.0  -1.0  1.0   0.0
 145.0   0.0  1.0   0.0  1.0   0.0  1.0  …  1.0   0.0  0.0   0.0  1.0   0.0
 147.0   0.0  0.0   0.0  1.0   0.0  0.0     0.0  -1.0  0.0   0.0  0.0   0.0
 136.0   0.0  0.0  -1.0  1.0   0.0  2.0     3.0  -2.0  0.0   0.0  0.0  -1.0
 144.0   0.0  1.0   0.0  1.0   0.0  0.0     2.0   0.0  1.0   0.0  1.0   0.0
 145.0   0.0  0.0   0.0  1.0   0.0  1.0     1.0   0.0  0.0   0.0  0.0  -1.0
   ⋮                           ⋮         ⋱        ⋮                    
 144.0  -2.0  0.0   0.0  1.0   0.0  0.0     0.0  -1.0  0.0   0.0  1.0  -1.0
 144.0   0.0  0.0  -1.0  1.0   0.0  1.0     0.0  -1.0  1.0   0.0  1

## Task 3: Similarity between the Sarcastic and Unsarcastic Bag of Words
Fill me in here.